# Getting started

This notebook walks through a small analysis of `data/example/sales.csv` using
**pandas**, **polars**, **numpy** and **matplotlib**.

Run a cell with **Shift + Enter**. If VS Code asks you to pick a kernel, choose
**Python Environments → .venv**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

## 1. Load data with pandas

`pd.read_csv` reads a CSV file into a *DataFrame* (a table). Paths are relative
to the root of the repository.

In [ ]:
df = pd.read_csv("data/example/sales.csv", parse_dates=["date"])
df.head()

In [ ]:
# Column types and missing values
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

## 2. Group and summarise with pandas

In [ ]:
by_region = (
    df.groupby("region")["revenue"]
    .agg(["count", "sum", "mean"])
    .sort_values("sum", ascending=False)
)
by_region

## 3. The same thing with polars

Polars is a newer, faster DataFrame library. Its syntax is different from
pandas but the ideas are the same. Use whichever you prefer.

In [ ]:
pl_df = pl.read_csv("data/example/sales.csv", try_parse_dates=True)

(
    pl_df.group_by("region")
    .agg(
        pl.len().alias("count"),
        pl.col("revenue").sum().alias("sum"),
        pl.col("revenue").mean().alias("mean"),
    )
    .sort("sum", descending=True)
)

## 4. Numerical work with numpy

In [ ]:
revenue = df["revenue"].to_numpy()

print(f"Mean:            {np.mean(revenue):,.2f}")
print(f"Median:          {np.median(revenue):,.2f}")
print(f"90th percentile: {np.percentile(revenue, 90):,.2f}")

## 5. Plot with matplotlib

In [ ]:
monthly = df.pivot_table(
    index=pd.Grouper(key="date", freq="MS"),
    columns="product",
    values="revenue",
    aggfunc="sum",
)

fig, ax = plt.subplots(figsize=(9, 5))
monthly.plot(ax=ax, marker="o")
ax.set_title("Monthly revenue by product")
ax.set_xlabel("")
ax.set_ylabel("Revenue (£)")
fig.tight_layout()
plt.show()

## 6. Save your results

Anything in `outputs/` stays out of git by default.

In [ ]:
by_region.to_csv("outputs/revenue_by_region.csv")
fig.savefig("outputs/monthly_revenue_by_product.png", dpi=150)